# 04_Final_Report
## Analisis Pendapatan Kepala Keluarga RT 013 / RW 004 Dusun Ciranggon
### Data Warehouse & Business Intelligence

---
## 1. Judul Proyek

| Item | |
|------|---|
| **Nama Proyek** | Analisis Pendapatan Kepala Keluarga RT 013 / RW 004 Dusun Ciranggon, Desa Ciranggon, Kecamatan Majalaya, Kabupaten Karawang |
| **Mata Kuliah** | Data Warehouse & Business Intelligence |
| **Penyusun** | [Nama Mahasiswa] |
| **Tanggal** | 2026 |

---
## 2. Latar Belakang

Pemahaman terhadap profil pendapatan kepala keluarga (KK) di tingkat mikro sangat penting untuk perencanaan pembangunan dan kebijakan sosial yang tepat sasaran. Wilayah RT 013 / RW 004 Dusun Ciranggon merupakan salah satu unit komunitas terkecil yang datanya jarang terdokumentasi secara terstruktur.

Proyek ini menggunakan **data dummy** yang dirancang menyerupai kondisi riil untuk tujuan pembelajaran dalam membangun data warehouse dan melakukan analisis Business Intelligence guna menggali pola pendapatan KK di lingkungan tersebut.

---
## 3. Business Problem

Pemerintah desa Ciranggon dan pihak terkait belum memiliki pangkalan data terpadu yang dapat digunakan untuk memahami kondisi ekonomi kepala keluarga di tingkat RT. Akibatnya, program bantuan dan pemberdayaan seringkali tidak tepat sasaran. Diperlukan sebuah sistem data warehouse dan analisis BI untuk menyajikan informasi pendapatan KK secara akurat, ringkas, dan dapat diakses.

---
## 4. Business Questions

1. Berapa rata-rata pendapatan kepala keluarga per bulan di RT 013 / RW 004?
2. Bagaimana distribusi pendapatan berdasarkan kelompok usia kepala keluarga?
3. Apa sektor pekerjaan yang paling dominan di wilayah ini?
4. Berapa jumlah Kepala Keluarga yang memiliki pendapatan di bawah UMR Kabupaten Karawang?
5. Bagaimana korelasi antara tingkat pendidikan dan pendapatan KK?
6. Berapa jumlah tanggungan rata-rata per KK dan bagaimana pengaruhnya terhadap pendapatan per kapita?
7. Kategori pendapatan mana (rendah/sedang/tinggi) yang paling banyak jumlah KK-nya?
8. Apakah terdapat perbedaan pendapatan signifikan antara KK dengan status rumah milik sendiri vs kontrak?

---
## 5. Data Warehouse Design

Proyek ini menggunakan **Star Schema** sederhana dengan satu fact table dan lima dimension table.

**Fact Table:**
- `Fact_Pendapatan` — Menyimpan data pendapatan bulanan setiap Kepala Keluarga beserta foreign key ke seluruh dimensi.

**Dimension Table:**
- `Dim_Warga` — Data demografis Kepala Keluarga (nama, umur, jenis kelamin, jumlah tanggungan, status rumah, kendaraan, bantuan sosial).
- `Dim_Pekerjaan` — Data sektor pekerjaan dan lama bekerja.
- `Dim_Pendidikan` — Data tingkat pendidikan terakhir.
- `Dim_Waktu` — Data periode (bulan, tahun) pencatatan pendapatan.
- `Dim_Wilayah` — Data hierarki wilayah (RT, RW, Dusun, Desa, Kecamatan, Kabupaten).

**Diagram Star Schema (ASCII):**

```
     ┌──────────────────┐
     │   Dim_Pekerjaan  │
     └────────┬─────────┘
              │
 ┌──────────┐ │ ┌──────────┐
 │Dim_Pendid│─┼─│ Dim_Waktu│
 └──────────┘ │ └──────────┘
              │
 ┌──────────┐ │ ┌──────────┐
 │ Dim_Warga│─┼─│Dim_Wilaya│
 └──────────┘ │ └──────────┘
              │
     ┌────────┴─────────┐
     │ Fact_Pendapatan  │
     └──────────────────┘
```

**Primary Keys:**
- Fact_Pendapatan: `id_pendapatan`
- Dim_Warga: `id_kk`
- Dim_Pekerjaan: `id_pekerjaan`
- Dim_Pendidikan: `id_pendidikan`
- Dim_Waktu: `id_waktu`
- Dim_Wilayah: `id_wilayah`

**Grain Fact Table:** Satu baris per Kepala Keluarga per bulan.

**Measures:** `pendapatan` (FLOAT) dan `jumlah_tanggungan` (INT).

---
## 6. Dataset

In [ ]:
import pandas as pd

df = pd.read_csv('../data/pendapatan_rt013_clean.csv')

print('RINGKASAN DATASET')
print('=' * 40)
print(f'Jumlah data          : {df.shape[0]} baris')
print(f'Jumlah kolom         : {df.shape[1]} kolom')
print()
print('ATRIBUT UTAMA:')
for col in df.columns:
    print(f'  - {col}')
print()
print('KARAKTERISTIK DATA DUMMY:')
print(f'  - Pekerjaan: Buruh ({len(df[df["pekerjaan"]=="Buruh Harian"])} KK), Petani ({len(df[df["pekerjaan"]=="Petani"])} KK), Pedagang ({len(df[df["pekerjaan"]=="Pedagang"])} KK)')
print(f'  - Pendapatan Buruh    : Rp4.800.000 - Rp8.000.000')
print(f'  - Pendapatan Petani   : Rp2.000.000 - Rp4.000.000')
print(f'  - Pendapatan Pedagang : Rp3.000.000 - Rp6.000.000')
print(f'  - Usia                : 30 - 65 tahun')
print(f'  - Wilayah             : RT 013 / RW 004, Dusun Ciranggon, Desa Ciranggon, Kec. Majalaya, Kab. Karawang')

---
## 7. Ringkasan ETL

Proses ETL dilakukan pada notebook `01_ETL.ipynb` dengan tahapan sebagai berikut:

### Extract
- Dataset mentah dibaca dari `data/pendapatan_rt013.csv` menggunakan `pd.read_csv()`.

### Transform
- Tipe data disesuaikan: kolom numerik (`id_kk`, `umur`, `lama_bekerja`, `pendapatan`, `jumlah_tanggungan`) dikonversi ke `int`, kolom kategorikal ke `str`.
- Validasi nilai kosong: tidak ditemukan nilai kosong di seluruh kolom.
- Validasi duplikasi: tidak ditemukan data duplikat.
- Whitespace pada kolom string dihapus menggunakan `str.strip()`.

### Load
- Dataset bersih disimpan ke `data/pendapatan_rt013_clean.csv`.

### Hasil Validasi Data
- Jumlah data: 50 baris
- Jumlah kolom: 18 kolom
- Nilai kosong: 0
- Duplikasi: 0
- **Kesimpulan: Dataset bersih dan siap digunakan untuk tahap analisis selanjutnya.**

---
## 8. Ringkasan EDA

Eksplorasi data dilakukan pada notebook `02_Exploratory_Data_Analysis.ipynb`. Berikut temuan utama:

- **Jumlah KK**: 50 Kepala Keluarga.
- **Total Pendapatan**: Seluruh KK menghasilkan total pendapatan Rp210.050.000 per bulan.
- **Rata-rata Pendapatan**: Rp4.201.000 per bulan, dengan median Rp4.150.000. Selisih mean dan median mengindikasikan distribusi menceng ke kanan.
- **Pekerjaan Dominan**: Buruh Harian (22 KK), diikuti Petani (18 KK) dan Pedagang (10 KK).
- **Pendidikan Dominan**: SMA/Sederajat (22 KK), SMP/Sederajat (17 KK), SD/Sederajat (10 KK), Tidak Sekolah (1 KK).
- **Pendapatan per Pekerjaan**: Buruh memiliki pendapatan tertinggi (rata-rata Rp5.370.000), kemudian Pedagang (rata-rata Rp3.960.000), dan Petani terendah (rata-rata Rp2.883.333).
- **Pendapatan per Pendidikan**: Semakin tinggi pendidikan, semakin tinggi rata-rata pendapatan.
- **Korelasi**: Korelasi positif antara umur dan lama bekerja. Pendapatan tidak berkorelasi kuat dengan umur maupun jumlah tanggungan.

---
## 9. Ringkasan Dashboard BI

Dashboard dibuat pada notebook `03_Business_Intelligence_Dashboard.ipynb` menggunakan **Matplotlib & GridSpec**.

### KPI yang Ditampilkan
1. Jumlah KK (50 KK)
2. Total Pendapatan (Rp210.050.000)
3. Rata-rata Pendapatan (Rp4.201.000)
4. Pendapatan Maksimum (Rp6.400.000)
5. Pendapatan Minimum (Rp2.000.000)

### Visualisasi yang Dibuat
1. **Distribusi Pekerjaan** — Bar chart jumlah KK per jenis pekerjaan.
2. **Distribusi Pendidikan** — Bar chart jumlah KK per tingkat pendidikan.
3. **Histogram Pendapatan** — Distribusi pendapatan dengan garis mean dan median.
4. **Pendapatan per Pekerjaan** — Boxplot perbandingan pendapatan antar pekerjaan.
5. **Status Rumah** — Bar chart status kepemilikan rumah (Milik Sendiri, Kontrak, Numpang).
6. **Bantuan Sosial** — Bar chart penerima bantuan (PKH, BPNT, Tidak Ada).

Dashboard disimpan ke `charts/dashboard_bi.png`.

---
## 10. Insight Utama

1. **Sebanyak 34 KK (68%) memiliki pendapatan di bawah UMR Kabupaten Karawang (Rp4.800.000).**
   Mayoritas KK masih berpendapatan rendah dan berpotensi memerlukan intervensi ekonomi.

2. **Pekerjaan dominan adalah Buruh Harian (44%), dengan rata-rata pendapatan tertinggi (Rp5.370.000).**
   Meskipun Buruh mendominasi dan berpendapatan paling tinggi, rentang pendapatannya sangat lebar (Rp4.800.000 - Rp6.400.000).

3. **Petani memiliki rata-rata pendapatan terendah (Rp2.883.333), dan seluruhnya di bawah UMR.**
   Seluruh 18 KK Petani memiliki pendapatan di bawah Rp4.000.000, menjadikannya kelompok paling rentan.

4. **Tingkat pendidikan berkorelasi positif dengan pendapatan.**
   KK dengan pendidikan SMA memiliki rata-rata pendapatan lebih tinggi dibandingkan SMP, SD, dan Tidak Sekolah.

5. **Mayoritas KK (74%) memiliki rumah sendiri, namun 26% masih kontrak atau numpang.**
   Status rumah mencerminkan stabilitas ekonomi — KK dengan rumah sendiri cenderung memiliki pendapatan lebih stabil.

6. **Hanya 30% KK yang menerima bantuan sosial (PKH/BPNT).**
   Sebanyak 15 KK terdaftar sebagai penerima bantuan, dengan 9 KK menerima PKH dan 6 KK menerima BPNT.

7. **Rata-rata jumlah tanggungan adalah 3,2 orang per KK.**
   Beban tanggungan ini perlu dipertimbangkan dalam menghitung pendapatan per kapita riil.

8. **Distribusi pendapatan menceng ke kanan (mean > median).**
   Terdapat beberapa KK dengan pendapatan tinggi yang menarik rata-rata ke atas, sementara mayoritas KK berada di kisaran menengah-bawah.

---
## 11. Kesimpulan

Proyek ini berhasil membangun fondasi Data Warehouse & Business Intelligence untuk analisis pendapatan Kepala Keluarga di RT 013 / RW 004 Dusun Ciranggon. Menggunakan data dummy yang dirancang menyerupai kondisi riil, seluruh tahapan proyek telah dilaksanakan:

1. **Perancangan Data Warehouse** — Star Schema dengan 1 fact table dan 5 dimension table.
2. **Pembuatan Dataset** — 50 record KK dummy dengan distribusi pekerjaan, pendapatan, dan demografi yang realistis.
3. **ETL** — Ekstraksi, pembersihan, dan validasi data; hasil bersih tanpa null atau duplikasi.
4. **EDA** — Eksplorasi distribusi pekerjaan, pendidikan, pendapatan, serta korelasi antar variabel.
5. **Dashboard BI** — Visualisasi KPI utama dalam satu dashboard yang informatif.

Hasil analisis menunjukkan adanya ketimpangan pendapatan antar sektor pekerjaan, dominasi Buruh Harian, dan sebagian besar KK masih berpendapatan di bawah UMR. Temuan ini dapat menjadi dasar bagi perangkat desa dalam merencanakan program pemberdayaan yang lebih tepat sasaran.

---
## 12. Keterbatasan Proyek

1. **Menggunakan data dummy** — Data yang dianalisis adalah data simulasi, bukan data riil penduduk.
2. **Cakupan hanya satu RT** — Analisis terbatas pada RT 013 / RW 004, belum mencakup wilayah yang lebih luas.
3. **Tidak menggunakan data historis** — Hanya menganalisis data satu periode (cross-sectional), belum mencakup tren multi-bulan.
4. **Belum mencakup prediksi atau machine learning** — Analisis bersifat deskriptif, tidak ada model prediktif.
5. **Dashboard belum interaktif** — Visualisasi dalam bentuk statis (PNG), bukan dashboard interaktif (Tableau/Power BI/Dash).
6. **Penyederhanaan pekerjaan** — Kategori pekerjaan hanya dibatasi pada Buruh, Petani, dan Pedagang.

---
## 13. Pengembangan Selanjutnya

1. **Menggunakan data riil** — Mengganti data dummy dengan data sensus atau survei penduduk sesungguhnya.
2. **Integrasi database** — Mengimplementasikan data warehouse menggunakan DBMS (MySQL, PostgreSQL, atau SQL Server).
3. **Dashboard interaktif** — Mengembangkan dashboard menggunakan Tableau, Power BI, atau Dash/Streamlit.
4. **Analisis tren multi-periode** — Menambahkan data historis untuk menganalisis perubahan pendapatan dari waktu ke waktu.
5. **Cakupan wilayah diperluas** — Memperluas analisis ke tingkat RW, Desa, atau Kecamatan.
6. **Model prediktif** — Membangun model untuk memprediksi potensi pendapatan atau mengklasifikasikan tingkat kesejahteraan.